In [10]:
!pip install google-play-scraper sastrawi pandas nltk

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 3.4 MB/s eta 0:00:00


### IMPORT LIBRARIES

In [11]:
import pandas as pd

In [13]:
from google_play_scraper import app, reviews, Sort, reviews_all

import pandas as pd
pd.options.mode.chained_assignment = None

import numpy as np
seed = 0
np.random.seed(seed)

### SCRAPING DATASET

In [14]:
# Scrapping dari Aplikasi Growtopia

from google_play_scraper import app, reviews_all, Sort

scrapreview = reviews_all(
    'com.rtsoft.growtopia',
    lang='id',
    country='id',
    sort=Sort.NEWEST,
    count=30000
)

df_review = pd.DataFrame(scrapreview)

print(f"Berhasil mengambil {df_review.shape[0]} baris ulasan dan {df_review.shape[1]} kolom.")

# Menampilkan 5 data teratas untuk memastikan
display(df_review.head())

df_review.to_csv('growtopia_review_raw.csv', index=False)
print("Data mentah berhasil disimpan sebagai 'growtopia_review_raw.csv'")

Berhasil mengambil 409580 baris ulasan dan 11 kolom.


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion
0,73e6d9f6-0e0a-4c9d-be8f-32361e1373a9,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,GAK JELAS LAGI FARMING TRUS COLLECT GEMS MALAH...,1,0,5.46,2026-04-30 06:19:58,None,NaT,5.46
1,819152c1-59a7-4e8e-9252-d0477894b39b,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,eror connection udah nunggu ampe 1jam tetep aj...,1,0,5.46,2026-04-30 01:52:53,None,NaT,5.46
2,fa1a7277-8ee3-4c06-8921-32f338561387,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,"mau login malah stuck loading, sudah hapus cac...",1,0,5.46,2026-04-29 21:54:50,None,NaT,5.46
3,87b94f04-2974-402e-b2cc-d431fc1bf499,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,login ae di persulit anj,1,0,5.46,2026-04-29 20:25:48,None,NaT,5.46
4,5424da01-48f0-412c-87e9-35d909aad49f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,error login,5,0,5.46,2026-04-29 19:14:22,None,NaT,5.46


Data mentah berhasil disimpan sebagai 'growtopia_review_raw.csv'


In [15]:
df_review.describe()

,score,thumbsUpCount,at,repliedAt
count,409580.000000,409580.000000,409580,43981
mean,3.142002,1.288989,2020-10-28 07:20:26.783681024,2021-11-18 00:31:28.507423232
min,0.000000,0.000000,2013-08-18 11:20:41,2017-08-09 12:21:43
25%,1.000000,0.000000,2018-06-25 04:35:17.500000,2020-11-04 06:21:40
50%,4.000000,0.000000,2021-04-01 12:30:48.500000,2021-09-26 07:16:54
75%,5.000000,0.000000,2023-07-16 03:38:21.500000,2023-07-08 20:24:52
max,5.000000,10106.000000,2026-04-30 06:19:58,2026-04-30 17:07:10
std,1.885261,33.629234,NaN,NaN


In [21]:
# Analisa per Kategori
print("=== SCORE <= 2 ===")
print(len(df_review[df_review['score']<=2]))

print("=== SCORE == 3 ===")
print(len(df_review[df_review['score']==3]))

print("=== SCORE >= 4 ===")
print(len(df_review[df_review['score']>=4]))

=== SCORE <= 2 ===
179628
=== SCORE == 3 ===
19320
=== SCORE >= 4 ===
210632


In [20]:
# Value tiap Score Rating
print(df_review['score'].value_counts())

score
5    192533
1    165374
3     19320
4     18099
2     14253
0         1
Name: count, dtype: int64


### DATA CLEANING

In [22]:
# 1. Menghapus baris yang memiliki nilai NaN
clean_df = df_review.dropna(subset=['content', 'score'])

# 2. Menghapus baris duplikat berdasarkan isi ulasan
clean_df = clean_df.drop_duplicates(subset=['content'])
print(f"Jumlah data setelah hapus NaN dan duplikat: {clean_df.shape[0]} baris\n")

# 3. Fungsi Pelabelan 3 Kelas Berdasarkan Rating
def pelabelan(score):
    if score <= 2:
        return 'Negatif'
    elif score == 3:
        return 'Netral'
    else:
        return 'Positif'

# Menerapkan fungsi pelabelan ke kolom baru 'label'
clean_df['label'] = clean_df['score'].apply(pelabelan)

print("Distribusi kelas SEBELUM di-balance:")
print(clean_df['label'].value_counts())

Jumlah data setelah hapus NaN dan duplikat: 316817 baris

Distribusi kelas SEBELUM di-balance:
label
Positif    156489
Negatif    142783
Netral      17545
Name: count, dtype: int64


### BALANCER
Section ini akan mengatur agar data bisa lebih seimbang, menggunakan undersampling.

In [26]:
df_negatif = clean_df[clean_df['label'] == 'Negatif']
df_netral = clean_df[clean_df['label'] == 'Netral']
df_positif = clean_df[clean_df['label'] ==  'Positif']

# Target pengurangan
target = int(round(len(df_netral) * 2.5, 0))

df_negatif_akhir = df_negatif.sample(n=min(target, len(df_negatif)), random_state=42)
df_positif_akhir = df_positif.sample(n=min(target, len(df_positif)), random_state=42)

# Gabungkan
cleaned_df = pd.concat([df_negatif_akhir, df_netral, df_positif_akhir])

# Shuffle
cleaned_df = cleaned_df.sample(frac=1, random_state=42).reset_index(drop=True)

print("Distribusi kelas SETELAH di-balance (Undersampling):")
print(cleaned_df['label'].value_counts())
print(f"\nTOTAL DATA SAAT INI: {cleaned_df.shape[0]} baris")

# Cek hasil akhir untuk memastikan tabel tetap panjang ke kanan
display(cleaned_df.head())

Distribusi kelas SETELAH di-balance (Undersampling):
label
Negatif    43862
Positif    43862
Netral     17545
Name: count, dtype: int64

TOTAL DATA SAAT INI: 105269 baris


,reviewId,userName,userImage,content,score,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt,appVersion,label
0,f91bea8c-a2dc-433e-8863-d1aacc61cac3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Gk bisa di delete,3,0,2.00,2015-11-22 06:22:51,None,NaT,2.00,Netral
1,b5c761c3-f280-4c74-b3e1-258301f22f77,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,MEngapa saya tidak bisa Connect ke Game ini?.....,1,0,2.10,2016-01-14 12:56:50,None,NaT,2.10,Negatif
2,55945b40-a549-4126-b2aa-9861222bf93a,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Game termantap.....,5,0,2.85,2018-06-01 11:25:47,None,NaT,2.85,Positif
3,a8acc00c-1fab-4622-b9da-a4514506ed2f,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Jangan rate 1 we,5,0,None,2023-07-18 12:32:59,None,NaT,None,Positif
4,78ef9c59-6f58-400d-840b-d85f445100e3,Pengguna Google,https://play-lh.googleusercontent.com/EGemoI2N...,Santai bro santai perjalanan kita masih panjang,3,0,None,2023-07-16 08:56:53,None,NaT,None,Netral


In [27]:
print("=== Info Dataset ===")
cleaned_df.info()

=== Info Dataset ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105269 entries, 0 to 105268
Data columns (total 12 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   reviewId              105269 non-null  object        
 1   userName              105269 non-null  object        
 2   userImage             105269 non-null  object        
 3   content               105269 non-null  object        
 4   score                 105269 non-null  int64         
 5   thumbsUpCount         105269 non-null  int64         
 6   reviewCreatedVersion  56696 non-null   object        
 7   at                    105269 non-null  datetime64[ns]
 8   replyContent          14554 non-null   object        
 9   repliedAt             14554 non-null   datetime64[ns]
 10  appVersion            56696 non-null   object        
 11  label                 105269 non-null  object        
dtypes: datetime64[ns](2), int64(2), objec

In [29]:
print("\n=== Pengecekan Missing Values (NaN) ===")
print(cleaned_df.isnull().sum())


=== Pengecekan Missing Values (NaN) ===
reviewId                    0
userName                    0
userImage                   0
content                     0
score                       0
thumbsUpCount               0
reviewCreatedVersion    48573
at                          0
replyContent            90715
repliedAt               90715
appVersion              48573
label                       0
dtype: int64


## SIMPAN DATA

In [30]:
cleaned_df.to_csv('growtopia_review_clean.csv', index=False)
print("Data berhasil disimpan sebagai 'growtopia_review_clean.csv'")

Data berhasil disimpan sebagai 'growtopia_review_clean.csv'
